**For buildings that overlaps, keep the one which has the highest height**

In the building dataset, some overlapping polygons represent the same building footprint but have different height values. To avoid duplicate representations, only the polygon with the greatest height should be retained.

**Pre-processing Steps**

1. Convert the building polygons to points using the **Feature to Point** tool.
2. Create two new integer fields:
   - `lat`
   - `long`
3. Use **Calculate Geometry** to populate the latitude and longitude coordinates of each point.

**Removing Duplicate Points**

In the following Python code, identify rows that share the same `lat` and `long` values. For each set of duplicate coordinates, retain only the row with the **maximum `height`** and discard all others.

In [1]:
from pathlib import Path

import pandas as pd
from dbfread import DBF

# Convert the external DBF table to CSV.
dbf_path = Path(r"C:\Users\percy\Documents\studies\visual_studio_code\big_files\wall_area_rescale\bldg_height_point.dbf")
out_csv = dbf_path.with_suffix(".csv")

if not dbf_path.exists():
    raise FileNotFoundError(f"DBF not found: {dbf_path}")

table = DBF(str(dbf_path), load=True, raw=False)
bldg_height_point = pd.DataFrame(iter(table))

# Keep only the requested fields and select the row with the highest height for each coordinate pair.
bldg_height_point_selected = bldg_height_point[["ID", "height", "lat", "long"]].copy()
idx = bldg_height_point_selected.groupby(["lat", "long"])['height'].idxmax()
bldg_height_point_selected = bldg_height_point_selected.loc[idx].sort_values(["lat", "long"]).reset_index(drop=True)

try:
    bldg_height_point_selected.to_csv(out_csv, index=False)
    saved_csv = out_csv
except PermissionError:
    saved_csv = dbf_path.with_name(f"{dbf_path.stem}_filtered.csv")
    bldg_height_point_selected.to_csv(saved_csv, index=False)

print(f"Loaded {len(table)} rows from {dbf_path}")
print(f"Saved {len(bldg_height_point_selected)} filtered rows to {saved_csv}")
print(f"Columns: {list(bldg_height_point_selected.columns)}")


Loaded 1250830 rows from C:\Users\percy\Documents\studies\visual_studio_code\big_files\wall_area_rescale\bldg_height_point.dbf
Saved 626814 filtered rows to C:\Users\percy\Documents\studies\visual_studio_code\big_files\wall_area_rescale\bldg_height_point.csv
Columns: ['ID', 'height', 'lat', 'long']


**Join the Height Table**
1. Join the processed CSV file containing the maximum building heights to the original building polygon shapefile using the appropriate unique identifier.
2. Export the joined layer as a new shapefile or feature class.
**Remove Lower-Height Polygons**
3. Delete polygons that do not contain any joined height values, as these correspond to duplicate building footprints with lower heights.
The result is `bldg_height_highest_filtered.shp`. Use this to generate roof area (2.3)
